<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-H3_ComfyUI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎬 MiniMax-H3 (ComfyUI runtime) — Video + Audio Generation (33B, INT8 quantized)

A Colab port of [MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) — a 33B parameter
video generation model that produces **video with synchronized audio** (ambience, foley,
speech). Supports text-to-video and image-to-video (first frame).

## How it works

MiniMax-H3 is run as a **headless ComfyUI subprocess** rather than via the stock diffusers
`ModularPipeline` integration:

1. **ComfyUI v0.30.1+ subprocess** on `127.0.0.1:8188`, launched with
   `--disable-pinned-memory --fp16-intermediates`. The pinned-memory flag is the difference
   between an OOM-kill on 32-53 GB host RAM and a 15-second video completing in ~25 min.
   See [`tonyd2wild/minimax-h3-local`](https://github.com/tonyd2wild/minimax-h3-local) for
   the verified recipe.

2. **`Comfy-Org/MiniMax-H3` weights** (`minimax_h3_fl2va_pruned_int8_convrot` diffusion
   transformer + `qwen3vl_32b_minimax_h3_nvfp4_awq` NVFP4 text encoder + `minimax_h3_video_vae_fp16`
   + `minimax_h3_audio_vae_fp32`).

3. **A pure-Python driver** that builds an API-format workflow JSON, POSTs `/prompt`,
   polls `/history/{prompt_id}`, downloads outputs via `/view`. The Gradio UI in STEP 4
   wraps the same driver for interactive use.

```
prompt + first-frame image → ComfyUI subprocess → video.mp4 + audio.wav
                                  (no GPU residency stacking — components swap on each step)
```

## ⚠️ License — Territory Restriction + MAU Cap

MiniMax H3 Community License:
- **Excludes:** EU, UK, South Korea, **and USA**
- **>1M MAU** requires separate commercial license
- Continuing past the header cell is your acceptance of the license

## Quick start

1. **Runtime → Change runtime type → GPU** (L4, A100, A100 80GB, or RTX 3090/4090)
2. Run **STEP 1** — git clones ComfyUI to Drive + installs torch cu130 + requirements
3. Run **STEP 2** — downloads Comfy-Org weights (~40.8 GB total) to Drive cache. First run: 30-60 min.
4. Run **STEP 3** — launches the ComfyUI subprocess and waits for `/system_stats` ready
5. Run **STEP 4** — opens the Gradio UI for t2va / fl2va
6. **STEP 5** keep-alive, **STEP 6** quick test, **STEP 7** batch

## Memory

| GPU | VRAM | Outcome |
|-----|------|---------|
| **A100 80GB** | 80 GB | int8_convrot + NVFP4 + `--disable-pinned-memory` |
| **A100 40GB** | 40 GB | int8_convrot + NVFP4 + `--disable-pinned-memory` |
| **L4 22GB** | 22 GB | int8_convrot + NVFP4 + `--disable-pinned-memory` |
| **RTX 3090/4090** | 24 GB | int8_convrot + NVFP4 + `--disable-pinned-memory` |

Total weight footprint on disk: ~40.8 GB. Fits in 53 GB Colab Pro+ host RAM.

## Outputs

```
<output_dir>/<prefix>_<timestamp>.mp4    # Video + synchronized stereo audio (32 kHz)
```

## Technical notes

- **ComfyUI v0.30.1+** for native H3 node support (`MiniMaxH3ImageToVideo`).
- **`--disable-pinned-memory`** is the single launch flag that prevents the host-RAM OOM-kill
  on 32-53 GB systems.
- **`--fp16-intermediates`** halves inter-node tensor sizes and is cheap insurance.
- **Drive cache**: weights live at `/content/drive/MyDrive/AEI_3D_Cache/H3_ComfyUI/weights/`
  and are symlinked into ComfyUI's `models/` subdirs. Subsequent runs skip the download.
- **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** reduces memory fragmentation on
  ComfyUI's intermediate buffers.

## Companion notebooks

- **`MiniMax-H3_Colab.ipynb`** — the diffusers attempt; left in the repo as documentation
  of the failure modes this notebook's `--disable-pinned-memory` flag sidesteps.
- **Wan2.2_Colab** — text/image-to-video (no audio)
- **Wan2.2_Animate_Colab** — character animation
- **GaussianGPT_Colab** — autoregressive 3D scene generation
- **InfiniSplat_Colab** — single-image 3DGS reconstruction


In [ ]:
#@title STEP 1 — Install ComfyUI v0.30.1+ and dependencies (Drive-persistent)

"""
• Mounts Google Drive for the weights cache + ComfyUI installation
• Clones ComfyUI v0.30.1+ to /content/drive/MyDrive/ComfyUI_H3/ on first run
• Installs torch 2.11.0+cu130 (L4 / T4 / A100 compatible)
• Installs ComfyUI's requirements.txt (transformers, tokenizers, safetensors, av, ...)
• Sets PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to reduce fragmentation
"""

import os, sys, subprocess, time, pathlib
from pathlib import Path

print('='*72)
print('MiniMax-H3 / ComfyUI — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path('/content/_h3_cache')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

COMFY_DIR = DRIVE_ROOT / 'ComfyUI_H3'
HF_CACHE  = DRIVE_ROOT / 'AEI_3D_Cache' / 'H3_ComfyUI'
OUT_DIR   = DRIVE_ROOT / 'AEI_3D_Out' / 'MiniMax-H3'
COMFY_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME']               = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE']  = str(HF_CACHE)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print(f'  Drive cache  : {HF_CACHE}')
print(f'  ComfyUI dir  : {COMFY_DIR}')
print(f'  Output dir   : {OUT_DIR}')

if not COMFY_DIR.joinpath('main.py').exists():
    print(f'  Cloning ComfyUI to {COMFY_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
else:
    print(f'  Reusing existing {COMFY_DIR}')

print('  Installing pytorch cu130 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'torch', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu130'], check=False)

print('  Installing ComfyUI requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-r', str(COMFY_DIR / 'requirements.txt')], check=False)

# The requirements pull in comfy-kitchen + comfy-aimdo which are no-op stubs for the
# MiniMax-H3 path. We don't need torchao for INT8 (ComfyUI handles it natively).

import torch as _torch_check
print(f'  torch        : {_torch_check.__version__}  (CUDA {_torch_check.version.cuda})')
if _torch_check.cuda.is_available():
    p = _torch_check.cuda.get_device_properties(0)
    print(f'  GPU          : {p.name}  ({p.total_memory / 1024**3:.1f} GB)')
    print(f'  Compute      : {p.major}.{p.minor}')
else:
    raise SystemExit('No GPU detected - ComfyUI needs CUDA.')


In [ ]:
#@title STEP 2 - Download Comfy-Org/MiniMax-H3 weights to Drive cache

import os, time, shutil
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi

# Comfy-Org/MiniMax-H3 layout (4 weights, ~40.8 GB on disk).
#   diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors  (19.5 GB)
#   text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors          (15.7 GB)
#   vae/minimax_h3_video_vae_fp16.safetensors                            ( 5.0 GB)
#   vae/minimax_h3_audio_vae_fp32.safetensors                            ( 0.6 GB)
# Total: ~40.8 GB on disk. The Drive cache keeps them across Colab sessions.
#
# Drive FUSE rejects symlinks and stalls on hardlinks of multi-GB files, so we
# full-copy into ComfyUI's models/ subdirs. We skip the copy when the dst already
# has the exact size from the Comfy-Org manifest (no spurious work on re-runs),
# and we verify sizes after the copy so a corrupted partial file gets caught
# here instead of as a `RuntimeError: shape '[...]' is invalid` deep in
# `comfy/utils.py load_safetensors` at sample time.

WEIGHTS_DIR = Path('/content/drive/MyDrive/AEI_3D_Cache/H3_ComfyUI/weights')
COMFY_DIR = Path('/content/drive/MyDrive/ComfyUI_H3')

PATTERNS = [
    'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',
    'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
    'vae/minimax_h3_video_vae_fp16.safetensors',
    'vae/minimax_h3_audio_vae_fp32.safetensors',
]

# Resolve expected sizes from the Hub manifest (one HTTP call, ~1s).
print('  Resolving expected sizes from Comfy-Org/MiniMax-H3 ...')
info = HfApi().repo_info('Comfy-Org/MiniMax-H3', files_metadata=True)
EXPECTED_BYTES = {sib.rfilename: sib.size for sib in info.siblings if sib.size is not None}

t0 = time.time()
weights = snapshot_download(
    repo_id='Comfy-Org/MiniMax-H3',
    local_dir=str(WEIGHTS_DIR),
    allow_patterns=PATTERNS,
)
print(f'  Weights cached at {weights} in {time.time() - t0:.0f}s')

DIFF = COMFY_DIR / 'models' / 'diffusion_models'
TXT  = COMFY_DIR / 'models' / 'text_encoders'
VAE  = COMFY_DIR / 'models' / 'vae'
for d in (DIFF, TXT, VAE):
    d.mkdir(parents=True, exist_ok=True)

def _expected_bytes(path_rel: str) -> int:
    return EXPECTED_BYTES.get(path_rel, -1)


def _wire(src: Path, dst: Path, expected_size: int):
    """Copy `src` to `dst` unless `dst` already matches `expected_size`.

    Skips the copy when the existing dst size matches the manifest exactly
    (cheap ~ms check, saves 8-25 min of Drive FUSE copy on re-runs).
    Re-runs that have a partial dst fall through to the full copy.
    Drives a 5-second heartbeat so the user sees progress.
    """
    expected_str = f'{expected_size/1024**3:.2f} GB'
    if dst.exists():
        existing = dst.stat().st_size
        if existing == expected_size:
            print(f'    OK  {dst.name}  ({expected_str} verified)', flush=True)
            return
        # Wrong size: usually a partial write from an interrupted earlier run.
        print(f'    -> {dst.name}: dst is {existing/1024**3:.2f} GB but expected {expected_str}; replacing.', flush=True)
        dst.unlink()
    elif expected_size <= 0:
        print(f'    ?? {dst.name}: no manifest size available; copying without verification.', flush=True)

    src_size_gb = src.stat().st_size / 1024**3
    print(f'    -> {dst.name}  ({src_size_gb:.1f} GB) ...', flush=True)
    t0 = time.time()
    copied = 0
    last_beat = time.time()
    with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
        while True:
            chunk = fsrc.read(64 * 1024 * 1024)  # 64 MB chunks
            if not chunk:
                break
            fdst.write(chunk)
            copied += len(chunk)
            if time.time() - last_beat > 5:
                pct = 100 * copied / (src_size_gb * 1024**3)
                rate = copied / (time.time() - t0) / 1024**2
                print(f'       {pct:5.1f}%  {rate:.0f} MB/s', flush=True)
                last_beat = time.time()
    print(f'    -> done in {time.time() - t0:.0f}s', flush=True)


def _verify():
    """Final sanity check: every dst matches the manifest size exactly.
    If anything is off, raise with a delete-and-retry message."""
    bad = []
    for pat in PATTERNS:
        dst = COMFY_DIR / 'models' / pat
        exp = _expected_bytes(pat)
        if not dst.exists():
            bad.append((pat, 0, exp))
            continue
        got = dst.stat().st_size
        if got != exp:
            bad.append((pat, got, exp))
    if bad:
        msg = 'Some weights do not match the Comfy-Org manifest.\n'
        for pat, got, exp in bad:
            msg += f'  {pat}: got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB\n'
        msg += ('Delete the bad files (e.g. '
                '`shutil.rmtree(\'/content/drive/MyDrive/ComfyUI_H3/models\')`) and re-run STEP 2.')
        raise SystemExit(msg)


_wire(WEIGHTS_DIR / 'diffusion_models' / 'minimax_h3_fl2va_pruned_int8_convrot.safetensors',
         DIFF / 'minimax_h3_fl2va_pruned_int8_convrot.safetensors',
         _expected_bytes('diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors'))
_wire(WEIGHTS_DIR / 'text_encoders' / 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
         TXT / 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
         _expected_bytes('text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'))
_wire(WEIGHTS_DIR / 'vae' / 'minimax_h3_video_vae_fp16.safetensors',
         VAE / 'minimax_h3_video_vae_fp16.safetensors',
         _expected_bytes('vae/minimax_h3_video_vae_fp16.safetensors'))
_wire(WEIGHTS_DIR / 'vae' / 'minimax_h3_audio_vae_fp32.safetensors',
         VAE / 'minimax_h3_audio_vae_fp32.safetensors',
         _expected_bytes('vae/minimax_h3_audio_vae_fp32.safetensors'))

_verify()

print('  Weights wired into ComfyUI/models/{diffusion_models,text_encoders,vae}/ (sizes verified against Comfy-Org/MiniMax-H3 manifest).')
total = 0
for d in (DIFF, TXT, VAE):
    for p in sorted(d.iterdir()):
        if p.suffix == '.safetensors':
            sz = p.stat().st_size / 1024**3
            total += sz
            print(f'    {sz:5.2f} GB  {d.name}/{p.name}')
print(f'  Total on ComfyUI side: {total:.2f} GB')


In [ ]:
#@title STEP 3 - Launch ComfyUI subprocess (--disable-pinned-memory)

import os, sys, time, signal, subprocess, urllib.request, urllib.error, json
from pathlib import Path

COMFY_DIR = Path('/content/drive/MyDrive/ComfyUI_H3')
COMFY_HOST = '127.0.0.1'
COMFY_PORT = 8188
COMFY_URL  = f'http://{COMFY_HOST}:{COMFY_PORT}'

# Pinned memory is the entire reason this notebook exists: the default ComfyUI policy pins
# ~90% of host RAM. With 53 GB of RAM the pinned ceiling is ~47 GB, and the diffusers runtime
# OOM-kills long before the model fits. --disable-pinned-memory is the single flag that makes
# this recipe work on 24-32 GB RAM (see tonyd2wild/minimax-h3-local).
# --fp16-intermediates halves inter-node tensors and is cheap insurance.
LAUNCH_CMD = [
    sys.executable, 'main.py',
    '--listen', COMFY_HOST,
    '--port', str(COMFY_PORT),
    '--disable-pinned-memory',
    '--fp16-intermediates',
    '--disable-api-nodes',
    '--output-directory', str(COMFY_DIR / 'output'),
    '--input-directory', str(COMFY_DIR / 'input'),
]
env = os.environ.copy()

subprocess.run(['pkill', '-9', '-f', 'ComfyUI_H3.*main.py'], check=False)
time.sleep(2)

(COMFY_DIR / 'output').mkdir(parents=True, exist_ok=True)
(COMFY_DIR / 'input').mkdir(parents=True, exist_ok=True)

print(f'  Launching: {" ".join(LAUNCH_CMD)}')
log_path = COMFY_DIR / 'comfyui.log'
log_f = open(log_path, 'wb')
proc = subprocess.Popen(
    LAUNCH_CMD,
    cwd=str(COMFY_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setsid,
)
print(f'  PID: {proc.pid}, log: {log_path}')

ready = False
deadline = time.time() + 600
while time.time() < deadline:
    try:
        with urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=2) as r:
            r.read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError, OSError):
        if proc.poll() is not None:
            print(f'  ComfyUI exited early with code {proc.returncode}. Last 40 log lines:')
            with open(log_path) as f:
                tail = f.read().splitlines()[-40:]
                for ln in tail:
                    print('   ', ln)
            raise SystemExit('ComfyUI failed to start.')
    time.sleep(2)

if not ready:
    proc.terminate()
    raise SystemExit(f'ComfyUI did not respond within 10 minutes. Tail of log:\n'
                     + '\n'.join(open(log_path).read().splitlines()[-20:]))

# Sanity check the model files BEFORE we hand the workflow off. ComfyUI's
# load_torch_file reads each tensor with `view(info["shape"])`, so a partial
# file whose metadata header says 33.5M elements but whose bytes are only
# 11.9M will throw `RuntimeError: shape '[...]' is invalid` deep inside the
# load. Catching it here instead of at denoise time is the difference
# between "fast feedback" and "45 min of waiting for the sampler to fail".
from huggingface_hub import HfApi as _HfApi
_HF_API = _HfApi()
_INFO = _HF_API.repo_info('Comfy-Org/MiniMax-H3', files_metadata=True)
_EXPECTED = {sib.rfilename: sib.size for sib in _INFO.siblings if sib.size is not None}
_PATTERNS = [
    'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',
    'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
    'vae/minimax_h3_video_vae_fp16.safetensors',
    'vae/minimax_h3_audio_vae_fp32.safetensors',
]
print('  Verifying ComfyUI/models file sizes against Comfy-Org/MiniMax-H3 manifest ...')
_size_bad = []
for pat in _PATTERNS:
    dst = COMFY_DIR / 'models' / pat
    exp = _EXPECTED.get(pat, -1)
    if not dst.exists():
        _size_bad.append((pat, 0, exp))
        print(f'    MISSING  {dst.name}  (expected {exp/1024**3:.2f} GB)')
        continue
    got = dst.stat().st_size
    if got != exp:
        _size_bad.append((pat, got, exp))
        print(f'    WRONG    {dst.name}  (got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB)')
    else:
        print(f'    OK       {dst.name}  ({exp/1024**3:.2f} GB)')
if _size_bad:
    proc.terminate()
    msg = ('ComfyUI-side weights do not match the Comfy-Org manifest. '
           'This usually means a Drive FUSE copy was interrupted before completion.\n\n')
    for pat, got, exp in _size_bad:
        msg += f'  {pat}: got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB\n'
    msg += ('\nFix: delete the bad models and re-run STEP 2:\n'
            '  !rm -rf /content/drive/MyDrive/ComfyUI_H3/models\n'
            '  # then re-run STEP 2 (it skips files that match the manifest size).\n')
    raise SystemExit(msg)

with urllib.request.urlopen(COMFY_URL + '/system_stats') as r:
    sysinfo = json.loads(r.read())
print(f'  ComfyUI ready: {sysinfo.get("system", {}).get("comfyui_version", "?")} on '
      f'{sysinfo.get("devices", [{}])[0].get("name", "?")}')

import builtins
builtins.H3_COMFY_PROC = proc
builtins.H3_COMFY_LOG = log_path
builtins.H3_COMFY_URL = COMFY_URL
builtins.H3_COMFY_DIR = COMFY_DIR
print('  Stored H3_COMFY_PROC / H3_COMFY_URL in builtins for STEP 4 access.')


In [ ]:
#@title STEP 4 - Gradio UI: t2v / fl2v with audio, runs through ComfyUI /prompt + /history

import os, json, time, uuid, shutil, urllib.request, urllib.parse, builtins, base64, io
from pathlib import Path

COMFY_URL  = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR  = getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))
PROC       = getattr(builtins, 'H3_COMFY_PROC', None)

import requests

def _poll_until_ready(url, timeout=10):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                r.read()
            return True
        except Exception:
            time.sleep(1)
    return False

if not _poll_until_ready(COMFY_URL + '/system_stats'):
    raise SystemExit('ComfyUI is not running. Re-run STEP 3.')

def _build_workflow(mode, prompt, width, height, length, steps, seed,
                    unet_name, clip_name, video_vae, audio_vae, first_frame=None):
    p = {}
    p["6"]  = {"class_type": "UNETLoader",   "inputs": {"unet_name": unet_name, "weight_dtype": "default"}}
    p["13"] = {"class_type": "CLIPLoader",   "inputs": {"clip_name": clip_name, "type": "minimax", "device": "default"}}
    p["11"] = {"class_type": "VAELoader",    "inputs": {"vae_name": video_vae}}
    p["24"] = {"class_type": "VAELoader",    "inputs": {"vae_name": audio_vae}}
    if mode == 'fl2v' and first_frame is not None:
        p["30"] = {"class_type": "LoadImage", "inputs": {"image": first_frame}}
    inputs = {"clip": ["13", 0], "vae": ["11", 0],
              "width": width, "height": height, "length": length, "prompt": prompt}
    if mode == 'fl2v' and first_frame is not None:
        inputs["first_frame"] = ["30", 0]
    p["104"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": inputs}
    p["16"] = {"class_type": "BasicGuider",   "inputs": {"model": ["6", 0], "conditioning": ["104", 0]}}
    p["17"] = {"class_type": "KSamplerSelect","inputs": {"sampler_name": "res_multistep"}}
    p["9"]  = {"class_type": "BasicScheduler","inputs": {"model": ["6", 0], "scheduler": "simple",
                                                          "steps": steps, "denoise": 1}}
    p["15"] = {"class_type": "RandomNoise",   "inputs": {"noise_seed": seed}}
    p["14"] = {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["15", 0], "guider": ["16", 0], "sampler": ["17", 0],
                          "sigmas": ["9", 0], "latent_image": ["104", 1]}}
    p["10"] = {"class_type": "VAEDecode",      "inputs": {"samples": ["14", 0], "vae": ["11", 0]}}
    p["23"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["14", 0], "vae": ["24", 0]}}
    p["91"] = {"class_type": "CreateVideo",    "inputs": {"images": ["10", 0], "audio": ["23", 0], "fps": 24}}
    p["92"] = {"class_type": "SaveVideo",      "inputs": {"video": ["91", 0],
                                                           "filename_prefix": f"video/H3_{mode}",
                                                           "format": "auto", "codec": "auto"}}
    return {"prompt": p}


def _upload_image(local_path):
    with open(local_path, 'rb') as f:
        files = {'image': (Path(local_path).name, f, 'image/png')}
        data  = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(COMFY_URL + '/upload/image', files=files, data=data, timeout=120)
    r.raise_for_status()
    return r.json()['name']


def _queue_workflow(wf, client_id):
    r = requests.post(COMFY_URL + '/prompt', json={"prompt": wf["prompt"], "client_id": client_id}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f'ComfyUI rejected workflow: {r.status_code} {r.text[:500]}')
    return r.json()['prompt_id']


def _wait_for_history(prompt_id, timeout=3600, poll=3):
    deadline = time.time() + timeout
    while time.time() < deadline:
        r = requests.get(f'{COMFY_URL}/history/{prompt_id}', timeout=10)
        if r.status_code == 200 and prompt_id in r.json():
            return r.json()[prompt_id]
        time.sleep(poll)
    raise TimeoutError(f'Workflow {prompt_id} did not finish within {timeout}s')


def _download_outputs(history_entry, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for node_id, node_out in history_entry.get('outputs', {}).items():
        for kind in ('videos', 'images', 'audio'):
            for entry in node_out.get(kind, []):
                fn = entry.get('filename')
                if not fn:
                    continue
                subdir = entry.get('type', 'output')
                src_url = f'{COMFY_URL}/view?filename={urllib.parse.quote(fn)}&type={subdir}'
                local = out_dir / fn
                with urllib.request.urlopen(src_url) as r, open(local, 'wb') as f:
                    shutil.copyfileobj(r, f)
                paths.append(str(local))
    return paths


import gradio as gr

OUTPUT_DIR = COMFY_DIR / 'output'

def _fmt(seconds):
    """Format a duration in seconds as `H:MM:SS` (or `MM:SS` under an hour).

    Used by the Gradio progress bar and the STEP 6 polling loop so the user can
    read elapsed time at a glance when a 124-frame H3 generation takes ~30-50 min.
    """
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

def run_minimax_h3(mode, prompt, first_frame,
                    width, height, length, steps, seed,
                    unet_name, clip_name, video_vae, audio_vae,
                    progress=gr.Progress(track_tqdm=False)):
    if not prompt or not prompt.strip():
        raise gr.Error('MiniMax-H3 needs a non-empty prompt.')
    while length % 17 != 5:
        length += 1

    client_id = str(uuid.uuid4())
    first_frame_name = None
    if mode == 'fl2v' and first_frame is not None:
        first_frame_name = _upload_image(first_frame)

    wf = _build_workflow(mode, prompt, int(width), int(height), int(length), int(steps), int(seed),
                          unet_name, clip_name, video_vae, audio_vae,
                          first_frame=first_frame_name)
    progress(0.05, desc=f'Queueing {mode} workflow ({width}x{height}, {length} frames, {steps} steps)...')
    prompt_id = _queue_workflow(wf, client_id)
    progress(0.1, desc=f'Queued as {prompt_id[:8]}. Polling ComfyUI queue...')

    started = time.time()
    while True:
        h = _wait_for_history(prompt_id, timeout=120)
        if h.get('status', {}).get('completed'):
            break
        if h.get('status', {}).get('error'):
            raise gr.Error('ComfyUI failed: ' + json.dumps(h['status'].get('messages', []), indent=2)[:2000])
        elapsed = time.time() - started
        progress(min(0.1 + 0.85 * (elapsed / 600), 0.95),
                 desc=f'Denoising in progress... {_fmt(elapsed)} elapsed ({(elapsed / max(int(steps),1)):.1f}s/step est.)')

    paths = _download_outputs(h, OUTPUT_DIR)
    progress(1.0, desc=f'Done in {_fmt(time.time() - started)}. Outputs: {[Path(p).name for p in paths]}')
    if not paths:
        raise gr.Error('Workflow finished but no outputs were produced.')
    video_path = next((p for p in paths if p.endswith(('.mp4', '.webm', '.mov'))), paths[0])
    report = (f'`{width}x{height}`, {length} frames ({length/24:.3f}s), {steps} steps - '
              f'time {time.time() - started:.0f}s - seed {seed} - mode `{mode}`')
    return video_path, report


CANVASES = [
    ('832 x 480 - 16:9',         832, 480),
    ('768 x 432 - 16:9 fast',    768, 432),
    ('640 x 360 - 16:9 cheapest', 640, 360),
    ('480 x 832 - 9:16',         480, 832),
    ('432 x 768 - 9:16 fast',    432, 768),
]
DEFAULT_CANVAS = '832 x 480 - 16:9'
DEFAULT_LENGTH = 124  # 5 s at 24 fps

UNET = 'minimax_h3_fl2va_pruned_int8_convrot.safetensors'
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

with gr.Blocks(title='MiniMax-H3 (ComfyUI)') as demo:
    gr.Markdown('# MiniMax-H3 - joint video + audio\n\nBackend: ComfyUI subprocess (--disable-pinned-memory). Weights: Comfy-Org/MiniMax-H3 int8_convrot + NVFP4.')
    gr.Markdown('### Welcome\n\nMiniMax-H3 generates 5-15 s of synchronized video + stereo audio from a prompt or first frame. The model is hosted by a ComfyUI subprocess running on 127.0.0.1:8188 (started in STEP 3).\n\n**First run:** keep the defaults (832 x 480, 124 frames = 5 s, 20 steps).')
    canvas_state = gr.State(DEFAULT_CANVAS)
    canvas_dims = gr.State((832, 480))
    with gr.Row():
        with gr.Column():
            mode = gr.Radio(choices=[('Text-to-video (t2va)', 't2va'),
                                      ('Image-to-video (fl2va)', 'fl2va')],
                            value='t2va', label='Mode',
                            info='t2va is the most reliable; fl2va needs an explicit first frame.')
            prompt = gr.Textbox(lines=6,
                                value=('Cinematic medium shot of a red fox trotting through a snowy pine forest at dawn, '
                                       'snow crunching underfoot, natural handheld micro-movement, shallow depth of field, '
                                       '35mm film grain.\n\noverall_soundscape: forest ambience, distant birdsong, '
                                       'footfalls on frozen snow.\nnon_diegetic_music: none.'),
                                label='Prompt',
                                info='Supports [Shot N] scene direction and Dialogue: lines. Inline dialogue is spoken by the audio decoder.')
            first_frame = gr.Image(label='First frame (fl2va only)', type='filepath', sources=['upload'])
            canvas_dd = gr.Dropdown(choices=[c[0] for c in CANVASES], value=DEFAULT_CANVAS, label='Canvas',
                                    info='Smaller canvas + shorter length = dramatically faster.')
            length = gr.Slider(56, 362, value=DEFAULT_LENGTH, step=1, label='Length (frames at 24 fps)',
                               info='Snapped to the 17n+5 grid. 124 ~ 5 s, 362 ~ 15 s.')
            with gr.Row():
                steps = gr.Slider(10, 40, value=20, step=1, label='Steps',
                                  info='20 is the upstream default; 24-28 marginally improves quality.')
                seed  = gr.Number(value=42, precision=0, label='Seed',
                                  info='0 = random.')
            run = gr.Button('Generate', variant='primary')
        with gr.Column():
            video_out = gr.Video(label='Video + soundtrack')
            report    = gr.Markdown()

    def _set_dims(label):
        for n, w, h in CANVASES:
            if n == label:
                return (w, h)
        return (832, 480)
    canvas_dd.change(_set_dims, canvas_dd, canvas_dims)

    run.click(
        run_minimax_h3,
        [mode, prompt, first_frame,
         gr.State(832), gr.State(480), length, steps, seed,
         gr.State(UNET), gr.State(CLIP), gr.State(VVAE), gr.State(AVAE)],
        [video_out, report],
    )

    def _welcome():
        return (
            'ComfyUI is running. Submit a workflow via the Gradio UI on '
            'http://127.0.0.1:7860 or POST to /prompt with an API-format workflow JSON.',
        )
    demo.load(_welcome, None, [report])

demo.queue(default_concurrency_limit=1).launch(share=False, inline=False, prevent_thread_lock=True, server_port=7860)
import builtins
builtins.H3_DEMO = demo
from IPython.display import clear_output as _clear
_clear()
print('\nGradio UI: http://127.0.0.1:7860 (open the Colab proxy URL above)')


In [ ]:
#@title STEP 5 - Keep-alive + session summary (ComfyUI status)

import os, time, json, urllib.request, builtins

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')

import IPython.display
display(IPython.display.Javascript("""
function KeepAlive() { console.log('Colab session kept alive at ' + new Date().toISOString()); }
setInterval(KeepAlive, 60000);
"""))

print('=' * 72)
print('MiniMax-H3 / ComfyUI session summary')
print('=' * 72)

try:
    with urllib.request.urlopen(URL + '/system_stats', timeout=5) as r:
        stats = json.loads(r.read())
    devs = stats.get('devices', [])
    for d in devs:
        free  = d.get('vram_free', 0) / 1024**3
        total = d.get('vram_total', 0) / 1024**3
        print(f'  GPU {d.get("name"):<24} {free:5.1f} GB free / {total:5.1f} GB total')
    print(f'  ComfyUI version  : {stats.get("system", {}).get("comfyui_version", "?")}')
    print(f'  Python version   : {stats.get("system", {}).get("python_version", "?")}')
    print(f'  Embedded at      : {URL}')
    print(f'  Log file         : {getattr(builtins, "H3_COMFY_LOG", "?")}')
    print(f'  Output dir       : {getattr(builtins, "H3_COMFY_DIR", "?")}/output')
    print()
    print('  ComfyUI subprocess is running. Submit a workflow via the Gradio UI on')
    print('  http://127.0.0.1:7860 or POST to /prompt with an API-format workflow JSON.')
except Exception as e:
    print(f'  [WARN] ComfyUI not reachable: {e}')


In [ ]:
#@title STEP 6 — Quick test (single video generation)

"""Stand-alone test. Generates one video with the parameters below."""
import os, sys, time, json, pathlib, random, uuid, shutil, urllib.request, urllib.parse, urllib.error, requests
from pathlib import Path
import builtins
from IPython.display import display, FileLink

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
OUT_DIR = Path(getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))) / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('='*72)
print('MiniMax-H3 / ComfyUI — single-video quick test')
print('='*72)

PROMPT = 'Cinematic medium shot of a red fox trotting through a snowy pine forest at dawn, snow crunching underfoot, natural handheld micro-movement, shallow depth of field, 35mm film grain.\n\noverall_soundscape: forest ambience, distant birdsong, footfalls on frozen snow.\nnon_diegetic_music: none.'  #@param {type:"string"}
IMAGE_PATH = ''  #@param {type:"string"}
LAST_IMAGE_PATH = ''  #@param {type:"string"}
CANVAS = "832 x 480 · 16:9"  #@param ['832 x 480 · 16:9', '768 x 432 · 16:9 fast', '640 x 360 · 16:9 cheapest', '480 x 832 · 9:16', '432 x 768 · 9:16 fast']
DURATION = 5  #@param {type:"slider", min:2, max:14, step:1}
STEPS = 20  #@param {type:"slider", min:10, max:40, step:1}
SEED = 42  #@param {type:"integer"}

# Canvas presets
CANVASES = {
    '832 x 480 · 16:9':          (832, 480),
    '768 x 432 · 16:9 fast':     (768, 432),
    '640 x 360 · 16:9 cheapest': (640, 360),
    '480 x 832 · 9:16':          (480, 832),
    '432 x 768 · 9:16 fast':     (432, 768),
}
WIDTH, HEIGHT = CANVASES[CANVAS]
LENGTH = DURATION * 24
while LENGTH % 17 != 5:
    LENGTH += 1
if SEED <= 0:
    SEED = random.randint(1, 2**31 - 1)
    print(f'  Random seed: {SEED}')

img_path = IMAGE_PATH.strip() or None
last_img_path = LAST_IMAGE_PATH.strip() or None

print(f'  Prompt      : {PROMPT[:60]}...')
print(f'  Canvas      : {WIDTH} x {HEIGHT}  ({LENGTH} frames = {LENGTH / 24:.3f}s at 24 fps)')
print(f'  Steps       : {STEPS}')
print(f'  Seed        : {SEED}')
print(f'  First frame : {img_path or "(none)"}')
print(f'  Last frame  : {last_img_path or "(none)"}')
print()

UNET = 'minimax_h3_fl2va_pruned_int8_convrot.safetensors'
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

# Build the workflow. fl2va if a first frame is given; else t2va.
inputs = {'clip': ['13', 0], 'vae': ['11', 0], 'width': WIDTH, 'height': HEIGHT,
          'length': LENGTH, 'prompt': PROMPT}
load_image_node = None
if img_path:
    # POST /upload/image to get a server-side filename
    with open(img_path, 'rb') as f:
        files = {'image': (Path(img_path).name, f, 'image/png')}
        data  = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(URL + '/upload/image', files=files, data=data, timeout=120)
    r.raise_for_status()
    server_name = r.json()['name']
    inputs['first_frame'] = ['30', 0]
    load_image_node = {
        '30': {'class_type': 'LoadImage', 'inputs': {'image': server_name}},
    }

nodes = {}
nodes.update(load_image_node or {})
nodes.update({
    '6':  {'class_type': 'UNETLoader',     'inputs': {'unet_name': UNET, 'weight_dtype': 'default'}},
    '13': {'class_type': 'CLIPLoader',     'inputs': {'clip_name': CLIP, 'type': 'minimax', 'device': 'default'}},
    '11': {'class_type': 'VAELoader',      'inputs': {'vae_name': VVAE}},
    '24': {'class_type': 'VAELoader',      'inputs': {'vae_name': AVAE}},
    '104':{'class_type': 'MiniMaxH3ImageToVideo', 'inputs': inputs},
    '16': {'class_type': 'BasicGuider',    'inputs': {'model': ['6', 0], 'conditioning': ['104', 0]}},
    '17': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'res_multistep'}},
    '9':  {'class_type': 'BasicScheduler', 'inputs': {'model': ['6', 0], 'scheduler': 'simple', 'steps': STEPS, 'denoise': 1}},
    '15': {'class_type': 'RandomNoise',    'inputs': {'noise_seed': SEED}},
    '14': {'class_type': 'SamplerCustomAdvanced',
           'inputs': {'noise': ['15', 0], 'guider': ['16', 0], 'sampler': ['17', 0],
                      'sigmas': ['9', 0], 'latent_image': ['104', 1]}},
    '10': {'class_type': 'VAEDecode',      'inputs': {'samples': ['14', 0], 'vae': ['11', 0]}},
    '23': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['14', 0], 'vae': ['24', 0]}},
    '91': {'class_type': 'CreateVideo',    'inputs': {'images': ['10', 0], 'audio': ['23', 0], 'fps': 24}},
    '92': {'class_type': 'SaveVideo',      'inputs': {'video': ['91', 0],
                                                        'filename_prefix': 'video/H3_quicktest',
                                                        'format': 'auto', 'codec': 'auto'}},
})

t0 = time.time()
r = requests.post(URL + '/prompt', json={'prompt': nodes, 'client_id': str(uuid.uuid4())}, timeout=60)
if r.status_code != 200:
    raise SystemExit(f'ComfyUI rejected the workflow: {r.status_code} {r.text[:1000]}')
prompt_id = r.json()['prompt_id']
print(f'  Queued: {prompt_id[:8]} ... polling /history')

last_report = 0
last_log_size = 0
log_path = Path(getattr(builtins, 'H3_COMFY_LOG', '/content/drive/MyDrive/ComfyUI_H3/comfyui.log'))
while True:
    h = requests.get(f'{URL}/history/{prompt_id}', timeout=10).json()
    if prompt_id in h:
        entry = h[prompt_id]
        if entry.get('status', {}).get('completed'):
            elapsed = time.time() - t0
            print(f'  Done in {_fmt(elapsed)} ({(elapsed / STEPS):.1f}s/step).')
            break
        if entry.get('status', {}).get('error'):
            raise SystemExit('ComfyUI failed: ' + json.dumps(entry['status'].get('messages', []), indent=2)[:2000])
    elapsed = time.time() - t0
    if time.time() - last_report > 30:
        last_report = time.time()
        # Tail the ComfyUI log: print any new lines so the user can see
        # "Sampling step N/M" or "VAE decoding..." messages from ComfyUI.
        new_lines = []
        if log_path.exists():
            cur_size = log_path.stat().st_size
            if cur_size > last_log_size:
                with log_path.open('rb') as f:
                    f.seek(last_log_size)
                    new_lines = f.read().decode('utf-8', errors='replace').splitlines()
                last_log_size = cur_size
        tail = ' | '.join(new_lines[-3:])[:200] if new_lines else ''
        print(f'    ... {_fmt(elapsed)} elapsed' + (f'  log: {tail}' if tail else ''))
    time.sleep(3)

paths = []
for node_id, node_out in entry.get('outputs', {}).items():
    for kind in ('videos', 'images', 'audio'):
        for out in node_out.get(kind, []):
            fn = out.get('filename')
            if not fn:
                continue
            subdir = out.get('type', 'output')
            url = f'{URL}/view?filename={urllib.parse.quote(fn)}&type={subdir}'
            local = OUT_DIR / fn
            with urllib.request.urlopen(url) as r, open(local, 'wb') as f:
                shutil.copyfileobj(r, f)
            paths.append(local)

print(f'\n  Outputs ({len(paths)}):')
for p in paths:
    print(f'    {p}')
video_path = next((str(p) for p in paths if str(p).endswith(('.mp4', '.webm', '.mov'))), str(paths[0]))
print(f'\n  Open: {video_path}')
display(FileLink(video_path))


In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list

"""
Advanced batch processor. Reads a JSON file containing a list of scenes, each with its
own prompt, optional keyframe images, canvas, duration, steps, and seed. The ComfyUI
subprocess is shared across scenes; each scene is a separate workflow.

JSON format (a list of objects):
```json
[
  {
    "prompt": "A red fox in a snowy pine forest at dawn",
    "image": "/content/drive/MyDrive/keyframes/fox_start.png",
    "last_image": "/content/drive/MyDrive/keyframes/fox_end.png",
    "canvas": "832 x 480 · 16:9",
    "duration": 5,
    "steps": 20,
    "seed": 42
  }
]
```

Fields (all optional except `prompt`):
  - prompt:    (required) text description of the scene
  - image:     (optional) path to first frame image
  - canvas:    (optional, default DEFAULT_CANVAS) one of the CANVASES keys
  - duration:  (optional, default DEFAULT_DURATION) seconds (2-14)
  - steps:     (optional, default DEFAULT_STEPS) inference steps (10-40)
  - seed:      (optional, default 0 = random) 0 or negative = random

A progress log is written to batch_log.jsonl so you can resume after a disconnect.
If the JSON file does not exist yet, this cell writes a starter template.
"""

import os, sys, time, json, pathlib, random, uuid, urllib.request, urllib.parse, requests, shutil, gc
from pathlib import Path
import builtins

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
OUT_DIR = Path(getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))) / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('='*72)
print('MiniMax-H3 / ComfyUI — Batch generation (JSON scene list)')
print('='*72)

BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/H3_ComfyUI/batch_scenes.json'  #@param {type:"string"}
DEFAULT_DURATION = 5  #@param {type:"slider", min:2, max:14, step:1}
DEFAULT_STEPS = 20  #@param {type:"slider", min:10, max:40, step:1}
DEFAULT_CANVAS = "832 x 480 · 16:9"  #@param ['832 x 480 · 16:9', '768 x 432 · 16:9 fast', '640 x 360 · 16:9 cheapest', '480 x 832 · 9:16', '432 x 768 · 9:16 fast']
SKIP_EXISTING = True  #@param {type:"boolean"}

CANVASES = {
    '832 x 480 · 16:9':          (832, 480),
    '768 x 432 · 16:9 fast':     (768, 432),
    '640 x 360 · 16:9 cheapest': (640, 360),
    '480 x 832 · 9:16':          (480, 832),
    '432 x 768 · 9:16 fast':     (432, 768),
}

json_path = Path(BATCH_JSON_PATH)
if not json_path.exists():
    json_path.parent.mkdir(parents=True, exist_ok=True)
    template = [
        {'prompt': 'A red fox in a snowy pine forest at dawn, slow dolly push-in, snow crunching underfoot.',
         'canvas': DEFAULT_CANVAS, 'duration': DEFAULT_DURATION, 'steps': DEFAULT_STEPS, 'seed': 42},
        {'prompt': 'A busy night market, neon signs reflecting in puddles, sizzling street food, ambient chatter.',
         'canvas': DEFAULT_CANVAS, 'duration': DEFAULT_DURATION, 'steps': DEFAULT_STEPS, 'seed': 7},
        {'prompt': 'A cellist playing a slow melody in an empty concert hall, warm stage lighting, distant applause at the end.',
         'canvas': DEFAULT_CANVAS, 'duration': DEFAULT_DURATION, 'steps': DEFAULT_STEPS, 'seed': 99},
    ]
    json_path.write_text(json.dumps(template, indent=2))
    print(f'  Created batch {json_path} with {len(template)} starter scenes.')
    print(f'  Edit it, then re-run STEP 7.')

with json_path.open() as f:
    scenes = json.load(f)
if not isinstance(scenes, list):
    raise SystemExit(f'Expected a JSON list, got {type(scenes).__name__}')
print(f'  Loaded {len(scenes)} scene(s) from {json_path}')

UNET = 'minimax_h3_fl2va_pruned_int8_convrot.safetensors'
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

results = []
total_start = time.time()
for i, sc in enumerate(scenes):
    prompt = sc.get('prompt', '').strip()
    if not prompt:
        print(f'  [{i}] SKIP: empty prompt')
        continue
    canvas_label = sc.get('canvas', DEFAULT_CANVAS)
    w, h = CANVASES.get(canvas_label, (832, 480))
    duration = int(sc.get('duration', DEFAULT_DURATION))
    length = duration * 24
    while length % 17 != 5:
        length += 1
    steps = int(sc.get('steps', DEFAULT_STEPS))
    seed = int(sc.get('seed', 0))
    if seed <= 0:
        seed = random.randint(1, 2**31 - 1)
    img_path = sc.get('image', '').strip() or None

    # SKIP_EXISTING — check whether a previously-completed run for this scene
    # already wrote the output. The simplest heuristic: filename starts with
    # `H3_batch_{i:03d}_` and lives in OUT_DIR.
    if SKIP_EXISTING:
        existing = list(OUT_DIR.glob(f'H3_batch_{i:03d}_*'))
        if existing:
            print(f'  [{i+1}/{len(scenes)}] SKIP (already exists: {existing[0].name})')
            results.append(str(existing[0]))
            continue

    print(f'\n  [{i+1}/{len(scenes)}] {w}x{h} {length}f {steps}steps seed={seed}')
    print(f'    {prompt[:80]}')

    inputs = {'clip': ['13', 0], 'vae': ['11', 0], 'width': w, 'height': h,
              'length': length, 'prompt': prompt}
    load_image_node = None
    if img_path:
        with open(img_path, 'rb') as f:
            files = {'image': (Path(img_path).name, f, 'image/png')}
            data  = {'type': 'input', 'overwrite': 'true'}
            r = requests.post(URL + '/upload/image', files=files, data=data, timeout=120)
        r.raise_for_status()
        inputs['first_frame'] = ['30', 0]
        load_image_node = {'30': {'class_type': 'LoadImage', 'inputs': {'image': r.json()['name']}}}

    nodes = {}
    nodes.update(load_image_node or {})
    nodes.update({
        '6':  {'class_type': 'UNETLoader',   'inputs': {'unet_name': UNET, 'weight_dtype': 'default'}},
        '13': {'class_type': 'CLIPLoader',   'inputs': {'clip_name': CLIP, 'type': 'minimax', 'device': 'default'}},
        '11': {'class_type': 'VAELoader',    'inputs': {'vae_name': VVAE}},
        '24': {'class_type': 'VAELoader',    'inputs': {'vae_name': AVAE}},
        '104':{'class_type': 'MiniMaxH3ImageToVideo', 'inputs': inputs},
        '16': {'class_type': 'BasicGuider',   'inputs': {'model': ['6', 0], 'conditioning': ['104', 0]}},
        '17': {'class_type': 'KSamplerSelect','inputs': {'sampler_name': 'res_multistep'}},
        '9':  {'class_type': 'BasicScheduler','inputs': {'model': ['6', 0], 'scheduler': 'simple', 'steps': steps, 'denoise': 1}},
        '15': {'class_type': 'RandomNoise',   'inputs': {'noise_seed': seed}},
        '14': {'class_type': 'SamplerCustomAdvanced',
               'inputs': {'noise': ['15', 0], 'guider': ['16', 0], 'sampler': ['17', 0],
                          'sigmas': ['9', 0], 'latent_image': ['104', 1]}},
        '10': {'class_type': 'VAEDecode',      'inputs': {'samples': ['14', 0], 'vae': ['11', 0]}},
        '23': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['14', 0], 'vae': ['24', 0]}},
        '91': {'class_type': 'CreateVideo',    'inputs': {'images': ['10', 0], 'audio': ['23', 0], 'fps': 24}},
        '92': {'class_type': 'SaveVideo',      'inputs': {'video': ['91', 0],
                                                            'filename_prefix': f'video/H3_batch_{i:03d}',
                                                            'format': 'auto', 'codec': 'auto'}},
    })

    t0 = time.time()
    r = requests.post(URL + '/prompt', json={'prompt': nodes, 'client_id': str(uuid.uuid4())}, timeout=60)
    if r.status_code != 200:
        print(f'    FAIL: {r.status_code} {r.text[:300]}')
        continue
    pid = r.json()['prompt_id']
    while True:
        h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
        if pid in h:
            entry = h[pid]
            if entry.get('status', {}).get('completed'):
                break
            if entry.get('status', {}).get('error'):
                print(f'    FAIL: {json.dumps(entry["status"].get("messages", []))[:300]}')
                continue
        time.sleep(5)
    elapsed = time.time() - t0
    for node_id, node_out in entry.get('outputs', {}).items():
        for v in node_out.get('videos', []):
            fn = v['filename']
            url = f'{URL}/view?filename={urllib.parse.quote(fn)}&type=output'
            local = OUT_DIR / fn
            with urllib.request.urlopen(url) as r2, open(local, 'wb') as f:
                shutil.copyfileobj(r2, f)
            results.append(str(local))
            print(f'    {local.name}  ({_fmt(elapsed)})')
    gc.collect()

total = time.time() - total_start
print(f'\nBatch complete: {len(results)}/{len(scenes)} clips, total {_fmt(total)} ({total / max(len(results),1):.0f}s/clip)')
for p in results:
    print(f'  {p}')


In [ ]:
#@title STEP 8 — Tail ComfyUI log (debugging aid)

import builtins
from pathlib import Path

LOG = Path(getattr(builtins, 'H3_COMFY_LOG', '/content/drive/MyDrive/ComfyUI_H3/comfyui.log'))
TAIL_LINES = 80  #@param {type:"slider", min:20, max:500, step:20}

if not LOG.exists():
    print(f'  Log not found: {LOG}')
else:
    lines = LOG.read_text(errors='replace').splitlines()
    n = min(TAIL_LINES, len(lines))
    print(f'=== last {n} lines of {LOG} ===')
    for ln in lines[-n:]:
        print(ln)
    print('=== end ===')
